# Fine-tuning GPT-2 for Funny Dish Name Generation

This notebook aims to train a GPT-2 model on custom restaurant data to generate funny and attractive dish names.

## Workflow

1. Prepare the dataset in plain text format  
2. Convert the data into the appropriate format for training  
3. Define the training arguments and train the model  
4. Save the trained model for future use


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorForLanguageModeling
from datasets import load_dataset
import os

In [ ]:
model_name= "gpt2"
model= GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

In [5]:
working_directory="/content/drive/MyDrive/transformers/Restaurant/Generate_Names"

In [6]:
os.getcwd()

'/content'

In [7]:
os.chdir(working_directory)

In [8]:
os.getcwd()

'/content/drive/MyDrive/transformers/Restaurant/Generate_Names'

In [9]:
!ls

fine_tuning_GPT2.ipynb	generate_namesipynb.ipynb  restaurant.txt


# Preparing training and evaluation datasets

In [ ]:
dataset= load_dataset("text", data_files={"data": "./restaurant.txt"} )

In [11]:
dataset["data"][0]

{'text': 'Style: funny | Cuisine: American | Ingredients: beef, bun, cheese | Name: The Big Moo-d Burger'}

In [12]:
dataset["data"][1]

{'text': 'Style: funny | Cuisine: American | Ingredients: chicken, batter, spices | Name: Cluck Norris Nuggets'}

In [13]:
split_dataset = dataset["data"].train_test_split(test_size=.2) # split data

In [14]:
split_dataset["train"]

Dataset({
    features: ['text'],
    num_rows: 122
})

In [15]:
train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

In [16]:
test_dataset["text"][0]

'Style: funny | Cuisine: American | Ingredients: bacon, egg, muffin | Name: Muffin Compares 2 U'

In [17]:
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

# Trainig Process

In [19]:
!ls

fine_tuning_GPT2.ipynb	generate_namesipynb.ipynb  restaurant.txt


In [20]:
working_directory

'/content/drive/MyDrive/transformers/Restaurant/Generate_Names'

In [21]:
# Prepare Training Arguments
save_output = working_directory + "/results"
training_args= TrainingArguments(
    output_dir= save_output,
    num_train_epochs= 32,
    per_device_train_batch_size= 2,
    per_device_eval_batch_size= 2,
    report_to= "none"
)

In [22]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer= tokenizer,
    mlm= False # mlm stands for masker language model , here it is equal to false because GPT2, unlike BERT, is a causal modeling language
)

In [23]:
trainer= Trainer(
    model= model,
    args= training_args,
    train_dataset = tokenized_train_dataset,
    eval_dataset= tokenized_test_dataset,
    data_collator= data_collator
)

In [ ]:
trainer.train()

In [ ]:
eval_results = trainer.evaluate()
print(eval_results)

In [ ]:
trainer.save_model("./final_model")

In [27]:
tokenizer.save_pretrained("./final_model")

('./final_model/tokenizer_config.json', './final_model/tokenizer.json')

In [28]:
!ls

final_model		generate_namesipynb.ipynb  results
fine_tuning_GPT2.ipynb	restaurant.txt


In [29]:
os.listdir("./final_model")

['config.json',
 'generation_config.json',
 'model.safetensors',
 'tokenizer_config.json',
 'tokenizer.json',
 'training_args.bin']